In [ ]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_order_reviews_table", "olist_order_reviews")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("order_reviews_table", "order_reviews_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_order_reviews_table_name = dbutils.widgets.get("raw_olist_order_reviews_table")

silver_schema = dbutils.widgets.get("silver_schema")
order_reviews_table_name = dbutils.widgets.get("order_reviews_table")

In [ ]:
raw_olist_order_reviews_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_order_reviews_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{order_reviews_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{order_reviews_table_name} (
            orderId STRING,
            reviewId STRING,
            reviewScore INT,
            reviewCommentTitle STRING,
            reviewCommentMessage STRING,
            reviewCreationTimestamp TIMESTAMP,
            reviewAnswerTimestamp TIMESTAMP,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
order_reviews_silver_df = (
    raw_olist_order_reviews_df
    .where(
        (col("order_id").rlike("^[0-9a-fA-F]{32}$")) & (col("review_id").rlike("^[0-9a-fA-F]{32}$")))
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("review_id").cast("string").alias("reviewId"),
        col("review_score").cast("int").alias("reviewScore"),
        col("review_comment_title").cast("string").alias("reviewCommentTitle"),
        col("review_comment_message").cast("string").alias("reviewCommentMessage"),
        to_timestamp(col("review_creation_date"), "yyyy-MM-dd HH:mm:ss").alias("reviewCreationTimestamp"),
        to_timestamp(col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("reviewAnswerTimestamp")
    )
    .withColumn("processedTimestamp", current_timestamp())
    .dropDuplicates(["orderId", "reviewId"])
)

In [ ]:
order_reviews_silver_df.createOrReplaceTempView("order_reviews_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{order_reviews_table_name} AS target
    USING order_reviews_silver_view AS source
    ON target.orderId = source.orderId AND target.reviewId = source.reviewId
    WHEN MATCHED THEN
        UPDATE SET 
            target.reviewScore = source.reviewScore,
            target.reviewCommentTitle = source.reviewCommentTitle,
            target.reviewCommentMessage = source.reviewCommentMessage,
            target.reviewCreationTimestamp = source.reviewCreationTimestamp,
            target.reviewAnswerTimestamp = source.reviewAnswerTimestamp,
            target.processedTimestamp = source.processedTimestamp
    WHEN NOT MATCHED THEN
        INSERT *
    """)